#  **비정형 문서 처리의 이해** (unstructured)

- **Unstructured**는 PDF, 이미지, HTML 등 **다양한 형식**의 비정형 데이터를 구조화된 형태로 변환하는 오픈소스 도구임

- 문서를 **제목**, **본문**, **표** 등 의미 있는 요소로 자동 분할하여 데이터 구조화를 지원

- **RAG 시스템** 구축을 위한 전처리 도구를 제공하며, 오픈소스와 API 서비스 버전으로 사용 가능

- **지원하는 문서 형식**

    - 텍스트: PDF, TXT, RTF, MD
    - 오피스: DOCX, XLSX, PPTX
    - 이메일: EML, MSG
    - 이미지: PNG, JPG, TIFF
    - 기타: HTML, XML, EPUB

- 오픈소스 버전의 **제한사항**

    - 프로덕션 환경에 최적화되지 않음
    - 고급 OCR 모델 미포함
    - by-page, by-similarity 청킹 전략 미지원
    - 보안 및 규정 준수 기능 제한
    - 문서 계층 구조 감지 기능 제한


---

## **Unstructrued 설치**

- pypi 설치: 

    - 모든 문서 타입을 사용하려면, `pip install "unstructured[all-docs]"` 명령어로 설치

    - 필수 **시스템 의존성**으로 **libmagic**, **poppler**, **libreoffice**, **pandoc**, **tesseract**가 요구됨

    - 설치: pip install "unstructured[all-docs]" / poetry add "unstructured[all-docs]"

- 공식 **Docker 이미지** 사용이 권장되며, 모든 의존성이 사전 설치되어 있어 즉시 사용 가능

    - 시스템 의존성 설치가 복잡할 수 있으므로 Docker 이미지 활용이 가장 효율적인 설치 방법

    - **도커 설치:**
        ```bash
        # pull the Docker Image
        docker pull downloads.unstructured.io/unstructured-io/unstructured:latest

        # create the container
        docker run -dt --name unstructured downloads.unstructured.io/unstructured-io/unstructured:latest

        # start a bash shell inside the running Docker container
        docker exec -it unstructured bash
        ```

    - 문서: https://docs.unstructured.io/open-source/installation

- **실습 환경 빌드**  

    - 설치 방법:
        ```bash
        # clone the repository
        git clone https://github.com/tsdata/unstructured-dev-vscode.git

        # change directory
        cd unstructured-dev-vscode

        # build and run the Docker container
        docker-compose up -d --build
        ```

    - 저장소: https://github.com/tsdata/unstructured-dev-vscode
  
    - [참고] onnxruntime 오류 발생 시:  
        ```bash
        # onnxruntime 설치 제거거
        pip uninstall onnxruntime

        # onnxruntime 재설치 (버전 지정)
        pip install onnxruntime==1.19.2
        ```    

- **VS Code Remote Development 설정**

    - 확장 프로그램에서 `Remote Development` 확장 프로그램을 설치 (Dev Containers)

    - 메뉴에서 `Attach to Running Container...` 선택

    - 실행 중인 컨테이너 중 하나를 선택하여 연결

- **Container 환경에서 프로젝트 환경 설정**

    - 메뉴에서 `Open Folder` 선택

    - 프로젝트 폴더를 선택하여 연결 (/workspace)

    - 커널 선택 (Python/Jupter 확장 프로그램 설치 후): venv 커널 선택 (/home/notebook-user/venv/bin/python)

    - pip install "langchain<1.0" "langchain-community<1.0" "langchain-huggingface<1.0" "langchain-openai<1.0" sentence_transformers faiss-cpu


---
## **환경 설정 및 준비**

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import logging

# 로깅 레벨 설정 
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('unstructured').setLevel(logging.ERROR)

---

## **주요 기능**

- 파티셔닝, 청킹, 정제의 세 가지 핵심 기능을 통해 효율적인 문서 전처리 파이프라인 구축 가능

### 1. **파티셔닝 (Partitioning)**

- **파티셔닝**은 비정형 문서를 구조화된 형태로 나누는 Unstructured의 주요 기능

- 비정형 데이터를 **구조화된 요소**로 변환하여 분석에 활용

`(1) 기본 파티셔너`

- **partition 함수**를 통해 PDF 문서의 파티셔닝 수행 가능

- 파티셔닝 방법은 **파일명 직접 지정** 또는 **파일 객체 사용** 두 가지 방식 제공

- 파티셔닝 결과는 `elements` 변수에 구조화된 형태로 저장됨

In [ ]:
from unstructured.partition.auto import partition

# 파일명으로 파티셔닝 (자동 파일 형식 감지)
elements = partition(filename="data/transformer.pdf")

print(f"파티션된 요소 수: {len(elements)}")
print("파티션된 요소:")
pprint(elements[:10])

In [ ]:
elements

In [ ]:
# 파일 객체로 파티셔닝
with open("data/transformer.pdf", "rb") as f:
    elements = partition(file=f)

print(f"파티션된 요소 수: {len(elements)}")

***Element:** 문서를 파티셔닝한 후 얻게 되는 기본 구성 단위

1. **텍스트 관련 Element**
    - `NarrativeText`: 본문 텍스트
    - `Title`: 제목
    - `ListItem`: 목록 항목
    - `UncategorizedText`: 분류되지 않은 텍스트

2. **구조적 Element**
    - `Table`: 표
    - `Header`: 헤더
    - `Footer`: 푸터
    - `PageBreak`: 페이지 구분

3. **특수 Element**
    - `Address`: 주소
    - `Formula`: 수식
    - `FigureCaption`: 그림 설명
    - `Image`: 이미지 메타데이터
    - `EmailAddress`: 이메일 주소
    - `CodeSnippet`: 코드 스니펫

In [ ]:
elements[0].to_dict()

In [ ]:
# Element 접근
for element in elements[:2]:
    # Element 유형 확인
    print(f"Type: {element.category}")
    
    # 텍스트 내용 접근
    print(f"Text: {element.text}")
    
    # 메타데이터 접근
    print(f"Page: {element.metadata.page_number}")
    print(f"Language: {element.metadata.languages}")

    # Dict로 변환
    print("Dict:")
    pprint(element.to_dict())

    print("-" * 100)

### **[실습]**

- pdf 문서(data/리비안_KR_without_table.pdf)를 기본 파티셔너를 적용해서 분석합니다. 
- 각 요소의 개수와 구성 내용을 출력하고 확인합니다. 

In [ ]:
# 여기에 코드를 작성하세요.

`(2) 문서 유형별 전용 파티셔너`

- 문서 형식별로 **전용 파티셔너** 제공으로 최적화된 처리 가능

- **PDF 문서**는 `partition_pdf` 함수로 특화된 파티셔닝 수행

- **HTML 문서**는 `partition_html` 함수를 통해 웹 문서에 최적화된 분할 처리

In [ ]:
# PDF 문서용
from unstructured.partition.pdf import partition_pdf
elements = partition_pdf(filename="data/transformer.pdf")

print(f"파티션된 요소 수: {len(elements)}")

In [ ]:
# HTML 문서용
from unstructured.partition.html import partition_html
elements = partition_html(filename="data/example.html")

print(f"파티션된 요소 수: {len(elements)}")

In [ ]:
elements[2].to_dict()

`(3) 파티셔닝 전략 선택`

- **파티셔닝 전략**은 `strategy` 매개변수를 통해 문서 특성에 맞게 설정 가능

- `auto`: 문서 특성에 따라 자동으로 최적 전략 선택 (기본값)
- `fast`: 텍스트 추출이 간단한 문서에 적합
- `hi_res`: 복잡한 레이아웃, 표, 이미지가 있는 문서에 적합
- `ocr_only`: 스캔된 문서나 이미지 기반 PDF에 적합

In [ ]:
# 단순 텍스트 추출이 가능한 문서
elements = partition_pdf(
    filename="data/transformer.pdf",
    strategy="fast"
)

print(f"파티션된 요소 수: {len(elements)}")

In [ ]:
# 처음 5개만 확인
elements[:5]

In [ ]:
# 첫 번째 요소 확인
elements[0].to_dict()

In [ ]:
# 고해상도 처리가 필요한 복잡한 문서
elements = partition_pdf(
    filename="data/transformer.pdf",
    strategy="hi_res",
    include_page_breaks=True
)

print(f"파티션된 요소 수: {len(elements)}")

In [ ]:
# 처음 5개만 확인
elements[:5]

In [ ]:
# 첫 번째 요소 확인
elements[0].to_dict()

In [ ]:
elements

In [ ]:
# 이미지 문서 
from unstructured.partition.pdf import partition_pdf


elements = partition_pdf(
    filename="data/transformer.pdf",
    strategy="ocr_only",
    include_page_breaks=True
)

print(f"파티션된 요소 수: {len(elements)}")

In [ ]:
# 처음 5개만 확인
elements[:5]

In [ ]:
# 첫 번째 요소 확인
elements[0].to_dict()

### **[실습]** 

- 2개의 각기 다른 pdf 문서를 PDF 전용 파티셔너를 적용해서 분석합니다. 
    - data/리비안_KR_without_table.pdf
    - data/리비안_KR_with_table.pdf
    
- 각기 다른 전략을 적용하고, 그 결과를 비교 분석합니다. 

In [ ]:
# 여기에 코드를 작성하세요.

### 2. **청킹 (Chunking)**

- 의미 단위로 구분된 청크는 **AI 모델의 문맥 이해도**를 높이는데 효과적

- RAG 시스템에서 고품질 검색과 정확한 응답 생성에 기여

- **기본 청킹 전략**

    - `chunk_elements` 함수는 연속된 요소들을 결합하여 최대 크기 내의 청크를 생성하는 방식으로 작동 

    - **크기 제한**은 `max_characters`(하드 최대값)와 `new_after_n_chars`(소프트 최대값) 두 가지 기준을 적용 

    - **테이블 처리**는 독립적으로 이루어지며, 크기 초과 시 **TableChunk**로 분할

    - 단일 요소가 최대 크기를 초과할 경우 **텍스트 분할**을 통해 여러 청크로 분할

In [ ]:
from unstructured.partition.auto import partition

# 기본 파티션 적용 
elements = partition(filename="data/transformer.pdf")

print(f"파티션된 요소 수: {len(elements)}")
print("파티션된 요소:")
pprint(elements[:10])

In [ ]:
from unstructured.chunking.basic import chunk_elements

chunks = chunk_elements(
    elements,
    max_characters=500,  # 청크의 하드 최대 크기
    new_after_n_chars=400,  # 청크의 소프트 최대 크기
    overlap=100,  # 청크 간 중복 문자 수
    overlap_all=True  # 모든 청크 간 중복 적용 여부
)

print(f"청크 수: {len(chunks)}")

# 각 청크의 문자 수 확인
chunk_lengths = [len(chunk.text) for chunk in chunks]
pd.Series(chunk_lengths).describe()

In [ ]:
# 처음 5개만 확인
chunks[:5]

In [ ]:
# 첫 번째 청킹 확인
chunks[0].to_dict()

In [ ]:
# 첫 번째 청킹 내용 확인
pprint(chunks[0].text)

In [ ]:
# 두 번째 청킹 확인
chunks[1].to_dict()

In [ ]:
# 두 번째 청킹 내용 확인
pprint(chunks[1].text)

### **[실습]** PDF 문서 청킹(Chunking) 전략 구현과 분석

**<요구 사항>**

1. PDF 문서를 청크(chunk)로 분할하는 함수 구현
    - PDF 파일 경로: "data/리비안_KR_without_table.pdf"

2. 다음 매개변수들을 조정하여 최소 2가지 다른 청킹 전략 실험:
   - max_characters
   - new_after_n_chars
   - overlap
   - overlap_all

3. 각 전략의 결과를 분석하여 다음 메트릭 계산:
   - 총 청크 수
   - 청크 길이의 기술 통계(평균, 표준편차, 최소, 최대)

In [ ]:
# 여기에 코드를 작성하세요.

### 3. **정제 (Cleaning)**

- **정제 기능**은 NLP 모델 사용 전에 데이터의 품질을 향상시키는 전처리 과정

- 기본적으로 제공되는 정제 함수들을 통해 **텍스트 데이터의 표준화**가 가능

- 사용자의 필요에 따라 **맞춤형 정제 규칙**을 추가하여 적용할 수 있음 

`(1) 기본 정제 함수 (clean)`

In [ ]:
from unstructured.cleaners.core import clean

# 여러 정제 옵션을 한번에 적용
cleaned_text = clean(
    "● An excellent ---   \n\n point.",
    bullets=True,      # 글머리 기호 제거
    extra_whitespace=True,  # 추가 공백 제거
    dashes=True,      # 대시 제거
    trailing_punctuation=True,  # 후행 문장부호 제거
    lowercase=True    # 소문자 변환
)

print(cleaned_text)

`(2) 특정 요소 정제`

In [ ]:
from unstructured.cleaners.core import clean_bullets

# 글머리 기호 제거
text = clean_bullets("● An excellent point!") 
print(text)

In [ ]:
from unstructured.cleaners.core import clean_extra_whitespace

# 추가 공백 제거
text = clean_extra_whitespace("ITEM 1A:     RISK-FACTORS")
print(text)

In [ ]:
from unstructured.cleaners.core import clean_dashes

# 대시 제거
text = clean_dashes("ITEM 1A: RISK-FACTORS–")
print(text)

In [ ]:
from unstructured.cleaners.core import remove_punctuation

# 문장부호 제거
text = remove_punctuation('"A lovely quote!"')
print(text)

`(3) 문단 재구성`

In [ ]:
from unstructured.cleaners.core import group_broken_paragraphs

text = """The big brown fox
was walking down the lane.

At the end of the lane, the
fox met a bear."""

# 줄바꿈으로 분리된 문단 결합
grouped_text = group_broken_paragraphs(text)

print(grouped_text)

`(4) 유니코드 문자 처리`

In [ ]:
from unstructured.cleaners.core import replace_unicode_quotes

# 유니코드 따옴표 변환
text = replace_unicode_quotes("\x93A lovely quote!\x94")

print(text)

In [ ]:
from unstructured.cleaners.core import clean_non_ascii_chars

# 비 ASCII 문자 제거
text = clean_non_ascii_chars("\x88This text contains ®non-ascii characters!●")

print(text)

`(5) 번역 기능`

In [ ]:
from unstructured.cleaners.translate import translate_text

# 자동 언어 감지 및 영어로 번역
english_text = translate_text("저는 대한민국 서울에 살고 있어요!")

print(english_text)

In [ ]:
# 특정 언어 지정 번역
english_text = translate_text("저는 대한민국 서울에 살고 있어요!", 
                            source_lang="ko", 
                            target_lang="en")

print(english_text)

`(6) Element 객체에 정제 적용`

In [ ]:
from unstructured.documents.elements import Text
from unstructured.cleaners.core import replace_unicode_quotes

text_element = Text(text="Philadelphia Eaglesâ\x80\x99 victory")
text_element.apply(replace_unicode_quotes)

print(text_element)

### **[실습]**

- 다음 예제 텍스트를 여러 방법을 적용하여 전처리합니다. 
- 그 결과를 비교 분석합니다.  

In [ ]:
text = """● 첫 번째 항목
    ■ 중첩된 항목
    
텍스트는    여러 개의     공백을 포함합니다.
또한 — 여러 종류의 ― 대시도 포함됩니다."""

In [ ]:
# 여기에 코드를 작성하세요.

---

## **RAG 구현**

- **RAG 구현**을 위해 Unstructured는 문서 파티셔닝, 청킹, 정제 기능을 **통합적으로** 제공함


### 1. **문서 전처리**

In [ ]:
from unstructured.partition.auto import partition
from unstructured.chunking.title import chunk_by_title
from langchain_core.documents import Document

# 문서 파티셔닝
elements = partition("data/transformer.pdf")

# 청킹
chunks = chunk_by_title(elements)

In [ ]:
# 메타데이터 확인
chunks[0].to_dict()

In [ ]:
# LangChain Document 변환
documents = []
for chunk in chunks:
    metadata = {}
    for key, value in chunk.metadata.to_dict().items():
        if key in ["filename", "page_number"]:
            metadata[key] = value

    documents.append(
        Document(page_content=chunk.text, metadata=metadata)
    )

# 처리 결과 확인
print(f"파티션 개수: {len(elements)}")
print(f"청킹 개수: {len(chunks)}")
print(f"문서 개수: {len(documents)}")

### 2. **문서 인덱싱**

In [ ]:
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings 

# 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(
    model_name="Alibaba-NLP/gte-multilingual-base",
    model_kwargs={"trust_remote_code": True}
    )

# 벡터 스토어 생성
db = FAISS.from_documents(documents, embeddings)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# 검색 쿼리 테스트
query = "Explain the working principle of transformer"
results = retriever.invoke(query)

# 결과 확인
for doc in results:
    print(doc.metadata)
    print(doc.page_content[:200])
    print()

### 3. **RAG 체인**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# LLM 설정
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.2,
    top_p=0.7,
)

# 프롬프트 템플릿 생성
system_prompt = """
당신은 인공지능 전문가입니다. 사용자의 질문에 대한 답변을 작성해야 합니다.

[작성 요령]
- 질문에 대한 답변을 간결하게 작성하세요.
- 질문에 대한 답변을 작성할 때, 가능한 한 사용자가 이해할 수 있도록 쉽게 설명하세요.
- 답변을 작성할 때, 사용자가 더 궁금해할만한 것을 더 알아보도록 유도하세요.
- 전문용어는 가능한 원어를 사용하고, 한국어를 사용할 경우에는 일반 명사를 제외한 용어는 소리나는 대로 함께 표기하세요. (예: Transformer(트랜스포머))
"""

human_prompt = """
반드시 주어진 문맥에 근거하여 답변해주세요. 근거가 부족하면 답변을 거부합니다. 
출처를 반드시 표시합니다. (출처: [문서 제목], [페이지 번호])

[문맥]
{context}

[질문]
{question}

이 질문에 대한 답변을 작성해주세요.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt), 
        ("human", human_prompt)
    ]
)

# 문서 포맷 함수
def format_document(doc):
    return f"**{doc.metadata['filename']} /page {doc.metadata['page_number']}**\n\n{doc.page_content}"

def format_documents(docs):
    return "\n\n".join([format_document(doc) for doc in docs])

# LCEL 생성
chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough()
    }) | prompt | llm | StrOutputParser()

# 챗봇 테스트
query = "트랜스포머 모델의 작동 원리를 설명해주세요."
response = chain.invoke(query) 

# 결과 확인
print(response)

### **[실습]** 

- 트랜스포머 논문(data/transformer.pdf)을 파티셔닝, 청킹, 정제 과정을 처리하여 벡터 저장소에 저장합니다. 
- RAG 체인을 구성하고, 검색 단계와 생성 단계를 구분하여 처리 결과를 분석합니다. 

In [ ]:
# 여기에 코드를 작성하세요.